# Notebook 04: Model Training

## 1: Load Feature Artifacts

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Load artifacts from Phase 4
event_features = joblib.load('../trained_models/event_features.pkl')
user_profiles = joblib.load('../trained_models/user_profiles.pkl')
interaction_matrix = joblib.load('../trained_models/interaction_matrix.pkl')
event_id_to_idx = joblib.load('../trained_models/event_id_to_idx.pkl')
idx_to_event_id = joblib.load('../trained_models/idx_to_event_id.pkl')
user_id_to_idx = joblib.load('../trained_models/user_id_to_idx.pkl')
idx_to_user_id = joblib.load('../trained_models/idx_to_user_id.pkl')
user_interaction_counts = joblib.load('../trained_models/user_interaction_counts.pkl')

events = pd.read_csv('../data/events_clean.csv')
registrations = pd.read_csv('../data/registrations_clean.csv')

print(f'Event features: {event_features.shape}')
print(f'User profiles: {len(user_profiles)}')
print(f'Interaction matrix: {interaction_matrix.shape}')

## 2: Temporal Train/Test Split

In [ ]:
registrations['registration_date'] = pd.to_datetime(registrations['registration_date'])

split_date = registrations['registration_date'].quantile(0.8)
train_regs = registrations[registrations['registration_date'] <= split_date]
test_regs = registrations[registrations['registration_date'] > split_date]

print(f'Split date: {split_date}')
print(f'Train: {len(train_regs)} registrations')
print(f'Test:  {len(test_regs)} registrations')

# Build ground truth: for each user in test, which events did they register for?
test_ground_truth = test_regs.groupby('user_id')['event_id'].apply(set).to_dict()
test_users = list(test_ground_truth.keys())
print(f'Test users: {len(test_users)}')

# Events the user already interacted with in training (to exclude from recs)
train_user_events = train_regs.groupby('user_id')['event_id'].apply(set).to_dict()

## 3: Train Content-Based Model

In [ ]:
from app.models.content_based import ContentBasedModel

cb_model = ContentBasedModel()
cb_model.fit(event_features)

print(f'Content-based model fitted')
print(f'Similarity matrix shape: {cb_model.similarity_matrix.shape}')

# Quick test: similar events to event 0
similar = cb_model.predict_similar(0, n=3)
print(f'\nEvents similar to "{events.iloc[0]["title"]}":')
for idx, score in similar:
    print(f'  [{score:.3f}] {events.iloc[idx]["title"]}')

## 4: Train Collaborative Model (SVD)

In [ ]:
from app.models.collaborative import CollaborativeModel

cf_model = CollaborativeModel(n_factors=20)
cf_model.fit(interaction_matrix)

print(f'Collaborative model fitted')
print(f'Predicted ratings shape: {cf_model.predicted_ratings.shape}')
print(f'Score range: [{cf_model.predicted_ratings.min():.2f}, {cf_model.predicted_ratings.max():.2f}]')

# Quick test: predictions for user 0
preds = cf_model.predict(0, n=3)
uid = idx_to_user_id[0]
print(f'\nTop predictions for user 0:')
for idx, score in preds:
    print(f'  [{score:.3f}] {events.iloc[idx]["title"]}')

## 5: Train Hybrid Model

In [ ]:
from app.models.hybrid import HybridRecommender

hybrid_model = HybridRecommender(content_weight=0.6, collab_weight=0.4)
hybrid_model.fit(event_features, interaction_matrix, n_factors=20)

print(f'Hybrid model fitted (content={hybrid_model.content_weight}, collab={hybrid_model.collab_weight})')
print(f'Cold-start threshold: {hybrid_model.COLD_START_THRESHOLD} interactions')

# Test with an active user
active_users = [(uid, cnt) for uid, cnt in user_interaction_counts.items() if cnt >= 5]
if active_users:
    uid, cnt = active_users[0]
    uidx = user_id_to_idx[uid]
    profile = user_profiles[uid]
    
    recs = hybrid_model.predict(profile, uidx, cnt, n=5)
    print(f'\nHybrid recommendations for user with {cnt} interactions:')
    for idx, score in recs:
        print(f'  [{score:.3f}] {events.iloc[idx]["title"]}')

## 6: Compare All 3 Models

In [ ]:
# Pick a user with enough interactions for meaningful comparison
uid, cnt = active_users[0]
uidx = user_id_to_idx[uid]
profile = user_profiles[uid]

print(f'Comparing models for user with {cnt} interactions\n')

cb_recs = cb_model.predict_for_user(profile, n=5)
cf_recs = cf_model.predict(uidx, n=5)
hy_recs = hybrid_model.predict(profile, uidx, cnt, n=5)

print(f'{"#":>2} {"Content-Based":40s} {"Collaborative":40s} {"Hybrid":40s}')
print('-' * 125)
for i in range(5):
    cb_title = events.iloc[cb_recs[i][0]]['title'][:37]
    cf_title = events.iloc[cf_recs[i][0]]['title'][:37]
    hy_title = events.iloc[hy_recs[i][0]]['title'][:37]
    print(f'{i+1:>2} {cb_title:40s} {cf_title:40s} {hy_title:40s}')

In [ ]:
# Test cold-start behavior
cold_users = [(uid, cnt) for uid, cnt in user_interaction_counts.items() if cnt < 3]

if cold_users:
    uid, cnt = cold_users[0]
    uidx = user_id_to_idx[uid]
    profile = user_profiles[uid]
    
    recs = hybrid_model.predict(profile, uidx, cnt, n=5)
    print(f'Cold-start user ({cnt} interactions) - hybrid falls back to content-based:')
    for idx, score in recs:
        print(f'  [{score:.3f}] {events.iloc[idx]["title"]}')
else:
    print('No cold-start users in this dataset (all have >= 3 interactions)')

## 7: Save Trained Models

In [ ]:
import os

model_dir = '../trained_models'

joblib.dump(hybrid_model, f'{model_dir}/hybrid_model.pkl')
joblib.dump(cb_model, f'{model_dir}/content_based_model.pkl')
joblib.dump(cf_model, f'{model_dir}/collaborative_model.pkl')

print('Saved models:')
for f in sorted(os.listdir(model_dir)):
    if f.endswith('.pkl'):
        size = os.path.getsize(f'{model_dir}/{f}') / 1024
        print(f'  {f:40s} {size:.1f} KB')